# Implementing an Earley Parser

**Contents**

1. [Setup and Imports](#Setup-and-Imports)
2. [Specification of the Grammar](#Specification-of-the-Grammar)
3. [Generating the Abstract Syntax Tree (AST)](#Generating-the-Abstract-Syntax-Tree-(AST))
   - [The Function `ast_to_rules`](#The-Function-ast_to_rules)
   - [The Function `read_grammar`](#The-Function-read_grammar)
4. [Earley's Algorithm](#Earley's-Algorithm)
   - [The Class `EarleyObject`](#The-Class-EarleyObject)
   - [The Class `Grammar`](#The-Class-Grammar)
   - [The Class `EarleyParser`](#The-Class-EarleyParser)
   - [The Function `fixpoint`](#The-Function-fixpoint)
   - [The Method `parse`](#The-Method-parse)
   - [The Method `completeAndPredict`](#The-Method-completeAndPredict)
   - [The Method `complete`](#The-Method-complete)
   - [The Method `predict`](#The-Method-predict)
   - [The Method `scan`](#The-Method-scan)
5. [Testing the Parser](#Testing-the-Parser)
   - [The Function `tokenize`](#The-Function-tokenize)
   - [The Function `test`](#The-Function-test)
6. [Further Tests](#Further-Tests)

## Setup and Imports

We import the module `re`, which is used by the function `tokenize`, and `Larkita`.  The parsers generated by
`Larkita` return the abstract syntax tree of their input as a nested tuple.

In [ ]:
import re

from IPython.display import Code, display

from larkita import Larkita

The file `simple.g` contains a grammar describing arithmetic expressions that only use the operators `+` and `*`.

In [ ]:
Code(filename='simple.g', language='larkita')

## Specification of the Grammar

`Larkita` allows us to define both lexical tokens and structural rules natively in a single file written in
*extended Backus-Naur Form* (EBNF).  Below is the grammar that `Larkita` will use to parse our context-free grammar specifications
(like `simple.g`).  These specifications are written in the notation of `Larkita`, too:  The alternatives of a
rule are separated by `"|"`, and every rule is terminated by a semicolon.
* The token `VARIABLE` recognizes a *syntactical variable*.  Syntactical variables start with a lower case letter.
* The token `TERMINAL` recognizes a *token*.  Tokens start with an uppercase letter.
* The token `LITERAL` recognizes a non-empty sequence of characters that is enclosed in double quotes.
* The literal `";"` at the end of the rule `rule` is the semicolon that terminates a rule of the grammar that is
  parsed.  The semicolon after the annotation `%ignore ":", "|", ";"` terminates the rule `rule` itself.
* The string literals of an alternative form the head of the tuple that is returned for this alternative.  As the
  literals `":"`, `"|"`, and `";"` carry no information, the annotation `%ignore ":", "|", ";"` removes them.  Then no literal is
  left and the name of the variable that is defined by the rule is used as head:  A rule yields the tuple `('rule', variable, body_1, ..., body_n)`,
  and a body yields the tuple `('body', item_1, ..., item_n)`.
* An alternative that consists of a single symbol returns the abstract syntax tree of this symbol.  Therefore, the
  rule `item` creates no tuple of its own.  A token is represented as a pair consisting of the name of the token and
  its text, for example `('LITERAL', '"+"')`.
* The `%ignore` directive tells the scanner to discard whitespace and newlines so they do not clutter our syntax tree.

In [ ]:
Code(filename='meta-grammar.g', language='larkita')

We generate the parser.  By default, `Larkita` generates a *canonical LR(1) parser* and writes a description of its
states into the file `meta-grammar.out`.

In [ ]:
grammar_parser = Larkita.open('meta-grammar.g', out_file='meta-grammar.out')

Let us check that Larkita was indeed able to generate a parser.

In [ ]:
with open('meta-grammar.out') as handle:
    report = handle.read()
print(report[:report.index('States\n')])

## Generating the Abstract Syntax Tree (AST)

The parser returns a nested tuple of the form `('start', rule_1, ..., rule_n)`.  Our custom `EarleyParser` expects the
grammar to be a list of rules, where each rule is a list of strings (e.g., `['expr', 'expr', '"+"', 'product']`).
Every token in the abstract syntax tree is a pair consisting of the name of the token and its text, and only the
text is needed.  The literals keep their double quotes, so that a literal like `"e"` can never be confused with
a variable like `e`.  

### The Function `ast_to_rules`

The function `ast_to_rules` drops the heads `'start'`, `'rule'`, and `'body'`:  For every body of a rule
defining the variable `a`, it creates the list that starts with `a` and continues with the items of the body.
Of every token, only its text is kept.

In [ ]:
def ast_to_rules(ast: tuple) -> list[list[str]]:
    _, *rules = ast
    return [ [variable] + [text for _, text in items]
             for _, (_, variable), *bodies in rules
             for _, *items in bodies ]

### The Function `read_grammar`

The function `read_grammar` reads the grammar file, displays it, parses it into a nested tuple, and converts it
into a list of rules.

In [ ]:
def read_grammar(file):
    with open(file, 'r') as handle:
        grammar_str = handle.read()
    display(Code(grammar_str, language='larkita'))
    ast = grammar_parser.parse(grammar_str)
    return ast_to_rules(ast)

In [ ]:
read_grammar('simple.g')

## Earley's Algorithm

### The Class `EarleyObject`

Given a context-free grammar $G = \langle V, \Sigma, R, s \rangle$ and a string $w = x_1x_2 \cdots x_n \in \Sigma^*$ of length $n$, 
an *Earley object* is a pair of the form
$$\langle a \rightarrow \alpha \bullet \beta, k \rangle$$
such that 
- $(a \rightarrow \alpha \beta) \in R\quad$  and
- $k \in \{0,1,\cdots,n\}$. 

In the literature, Earley objects are also called *Earley items*.

The class `EarleyObject` represents a single *Earley object*.  It is a *data class*, i.e. a class whose main
purpose is to store data.  A class becomes a data class if its definition is preceded by the *decorator*
`@dataclass`, which is imported from the module `dataclasses` of the standard library.  In the body of a data class,
we only declare the *fields* of the class, i.e. its member variables, together with their types:
- `mVariable` is the variable $a$,
- `mAlpha` is $\alpha$, a tuple of strings,
- `mBeta` is $\beta$, a tuple of strings, and
- `mIndex` is $k$.

From these declarations, the decorator `@dataclass` generates the following methods:
- The *constructor* `__init__` has one parameter for every field.  The parameters occur in the order in which the
  fields are declared, and the constructor stores the arguments in the fields.  For example, the call
  `EarleyObject('expr', ('expr', '"+"'), ('product',), 0)` creates the Earley object
  $\langle \texttt{expr} \rightarrow \texttt{expr}\; \texttt{"+"} \bullet \texttt{product}, 0 \rangle$.
- The method `__eq__` compares two objects field by field:  Two Earley objects are equal if and only if they
  agree in all four fields.  Python derives the operator `!=` from the method `__eq__`.

The arguments of the decorator control which further methods are generated:
- `frozen=True` makes the objects *immutable*:  After an Earley object has been created, its fields cannot be
  changed.  An assignment to a field raises the exception `FrozenInstanceError`.  As the fields of a frozen
  object never change, its hash value cannot change either.  Therefore, the decorator also generates the method
  `__hash__`, which computes the hash value of an object from the values of all its fields.  We need this method,
  since Earley's algorithm stores Earley objects in sets and Python requires the elements of a set to have a hash
  value.  For the same reason, `mAlpha` and `mBeta` are tuples:  Lists can be changed and therefore have no hash
  value.
- By default, the decorator also generates a method `__repr__`, which lists the names and the values of all
  fields.  The argument `repr=False` suppresses this method, since we define our own method `__repr__`, which
  writes an Earley object in the notation used above.

Without the decorator, we would have to write the constructor and the methods `__eq__` and `__hash__` ourselves.

The methods that are only declared in the class body, i.e. with the body `...`, are implemented in the
cells below and attached to the class afterwards.  These declarations tell the type checker which
methods the class provides.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, repr=False)
class EarleyObject():
    mVariable: str
    mAlpha:    tuple[str, ...]
    mBeta:     tuple[str, ...]
    mIndex:    int

    def __repr__(self) -> str:
        rhs = ' '.join(self.mAlpha + ('•',) + self.mBeta)
        return f'<{self.mVariable} → {rhs}, {self.mIndex}>'

    def isComplete(self) -> bool: ...
    def sameVar(self, c: str) -> bool: ...
    def scan(self, T: str) -> bool: ...
    def nextVar(self) -> str | None: ...
    def moveDot(self) -> EarleyObject: ...

Given an Earley object `self`, the method `isComplete` checks whether the Earley object `self` has the form
$$\langle a \rightarrow \alpha \bullet, k \rangle,$$
i.e. whether the $\bullet$ is at the end of the grammar rule.

In [ ]:
def isComplete(self):
    return self.mBeta == ()

EarleyObject.isComplete = isComplete
del isComplete

The method `sameVar(self, c)` checks whether the symbol following the dot is the same as the variable 
given as argument, i.e. `sameVar(self, c)` returns `True` if `self` is an Earley object of the form
$$\langle a \rightarrow \alpha \bullet c\beta, k \rangle.$$

In [ ]:
def sameVar(self, c: str) -> bool:
    return len(self.mBeta) > 0 and self.mBeta[0] == c

EarleyObject.sameVar = sameVar 
del sameVar

The method `scan(self, T)` checks whether the symbol following the dot matches the token `T`, 
i.e. `scan(self, T)` returns `True` if `self` is an Earley object of the form
$$\langle a \rightarrow \alpha \bullet T\beta, k \rangle.$$
The argument $T$ can either be the name of a token or a literal.

In [ ]:
def scan(self, T: str) -> bool:
    return len(self.mBeta) > 0 and self.mBeta[0] == T

EarleyObject.scan = scan
del scan

Given an Earley object, the method `nextVar` returns the name of the variable following the dot.  If there is no variable following the dot, the method returns `None`.  The method can distinguish variables from tokens because only the names of variables start with a lower case letter:  The names of tokens start with an upper case letter and literals start with a double quote.

In [ ]:
def nextVar(self) -> str | None:
    if len(self.mBeta) > 0:
        var = self.mBeta[0]
        if var[0].islower():
            return var
    return None

EarleyObject.nextVar = nextVar
del nextVar

The method `moveDot(self)` moves the $\bullet$ in the Earley object `self`, where `self` has the form 
$$\langle a \rightarrow \alpha \bullet \beta, k \rangle$$
over the next variable, token, or literal in $\beta$.  It assumes that $\beta$ is not empty.

In [ ]:
def moveDot(self) -> EarleyObject:
    return EarleyObject(self.mVariable, 
                        self.mAlpha + (self.mBeta[0],), 
                        self.mBeta[1:], 
                        self.mIndex)

EarleyObject.moveDot = moveDot
del moveDot

### The Class `Grammar`

The class `Grammar` represents a context-free grammar.  It stores a list of the rules of the grammar.
Each grammar rule of the form
$$ a \rightarrow \beta $$
is stored as a list that starts with $a$ and continues with the symbols of $\beta$.  The start symbol is assumed to be the variable on the left hand side of
the first rule. To distinguish syntactical variables from tokens, variables start with a lower case letter,
while tokens either start with an upper case letter or they are literals like `"+"`, which keep their double
quotes.

The methods that are only declared in the class body, i.e. with the body `...`, are implemented in the
cells below and attached to the class afterwards.  These declarations tell the type checker which
methods the class provides.

In [ ]:
class Grammar():
    def __init__(self, Rules: list[list[str]]):
        self.mRules = Rules

    def startObject(self) -> EarleyObject: ...
    def finishObject(self) -> EarleyObject: ...
    def startVar(self) -> str: ...

The method `startObject` returns the Earley object
$$ \langle\hat{s} \rightarrow \bullet s, 0\rangle $$
where $s$ is the start variable of the given grammar and $\hat{s}$ is the new variable `Start`.

In [ ]:
def startObject(self):
    return EarleyObject('Start', (), (self.startVar(),), 0)

Grammar.startObject = startObject
del startObject

The method `finishObject` returns the Earley object
$$ \langle\hat{s} \rightarrow s \bullet, 0\rangle $$
where $s$ is the start variable of the given grammar and $\hat{s}$ is a new variable.

In [ ]:
def finishObject(self):
    return EarleyObject('Start', (self.startVar(),), (), 0)

Grammar.finishObject = finishObject
del finishObject

The method `startVar` returns the start variable of the grammar.  It is assumed that
the first rule of the grammar defines the start variable.

In [ ]:
def startVar(self) -> str:
    return self.mRules[0][0]

Grammar.startVar = startVar
del startVar

### The Class `EarleyParser`

The class `EarleyParser` implements the [parsing algorithm of Jay Earley](https://en.wikipedia.org/wiki/Earley_parser).
The class maintains the following member variables:
- `mGrammar` is the grammar that is used to parse the given token string.
- `mString` is the list of tokens and literals that has to be parsed.

   As a hack, the first element of this list is `None`.  
   Therefore, `mString[i]` is the $i^\textrm{th}$ token.
- `mStateList` is a list of sets of *Earley objects*.  If $n$ is the length of the given token string
  (excluding the first element `None`), then $Q_i = \texttt{mStateList}[i]$. 
  The idea is that the set $Q_i$ is the set of those *Earley objects* that the parser could be in 
  when it has read the tokens `mString[1]`, $\cdots$,  `mString[i]`.  $Q_0$ is initialized as follows:
  $$ Q_0 = \bigl\{\langle\hat{s} \rightarrow \bullet s, 0\rangle\bigr\}. $$
  
The *Earley objects* are interpreted as follows: If we have
$$ \langle c \rightarrow \alpha \bullet \beta, k\rangle \in Q_i, $$
then we know the following:
- After having read the tokens `mString[:k+1]` the parser tries to parse the variable $c$
  in the token string `mString[k+1:]`.
- After having read the token string `mString[k+1:i+1]` the parser has already recognized $\alpha$
  and now needs to recognize $\beta$ in the token string `mString[i+1:]` in order to parse the variable $c$.

The methods that are only declared in the class body, i.e. with the body `...`, are implemented in the
cells below and attached to the class afterwards.  These declarations tell the type checker which
methods the class provides.

In [ ]:
class EarleyParser():
    def __init__(self, grammar, TokenList):
        self.mGrammar   = grammar 
        self.mString    = [None] + TokenList  # hack so mString[1] is the first token
        self.mStateList = [set() for _ in range(len(TokenList)+1)] 
        self.mStateList[0] = { self.mGrammar.startObject() }

    def parse(self) -> None: ...
    def completeAndPredict(self, Qi: set[EarleyObject], i: int) -> set[EarleyObject]: ...
    def complete(self, Qi: set[EarleyObject], i: int)           -> set[EarleyObject]: ...
    def predict(self, Qi: set[EarleyObject], i: int)            -> set[EarleyObject]: ...
    def scan(self, i: int) -> None: ...

### The Function `fixpoint`

The method `parse`, which is defined below, computes every set $Q_i$ with a *fixed-point iteration*.  The function
`fixpoint` implements this iteration.  It is the same function as in the notebook
`Chapter-04-05/01-NFA-2-DFA.ipynb`:  Given a function $f$ and a start value $x$, it computes the values
$$ x,\; f(x),\; f\bigl(f(x)\bigr),\; f\Bigl(f\bigl(f(x)\bigr)\Bigr),\; \cdots $$
until it finds a value $y$ such that $f(y) = y$, i.e. a *fixed point* of $f$.  The additional arguments `args` are
passed to `f` in every call.  The method `parse` uses them to pass the index $i$ of the set $Q_i$ to `f`.  The
notebook `01-NFA-2-DFA.ipynb` explains the type annotation of `fixpoint`.

In [ ]:
from collections.abc import Callable

In [ ]:
def fixpoint[T, *Ts](f: Callable[[T, *Ts], T], x: T, *args: *Ts) -> T:
    while True:
        y = f(x, *args)
        if y == x:
            return x
        x = y

### The Method `parse`

The method `parse` implements Earley's algorithm.  For all sets 
$Q_0$, $Q_1$, $\cdots$, $Q_n$ we proceed as follows:
- We apply the *completion* operation and the *prediction* operation until no more Earley objects are added to
  $Q_i$.  The result is the smallest set that contains the Earley objects that are already in $Q_i$ and that is
  closed under completion and prediction.  It is computed by a fixed-point iteration that starts with the current
  value of $Q_i$:  For $i = 0$, this is the set containing the start object, otherwise it is the set of those Earley
  objects that the scanning operation has added to $Q_i$.  A single step of this iteration is computed by the method
  `completeAndPredict`, which is defined below.
- Finally, the *scanning* operation is applied to $Q_i$.  This operation adds
  Earley objects to the set $Q_{i+1}$.

After $Q_i$ has been computed, we proceed to process $Q_{i+1}$.
Parsing is successful iff
$$ \langle\hat{s} \rightarrow s \bullet, 0\rangle \in Q_n $$

In [ ]:
def earley_parse(self):
    "run Earley's algorithm"
    n = len(self.mString) - 1 # mString[0] = None
    for i in range(0, n+1):
        self.mStateList[i] = fixpoint(self.completeAndPredict, 
                                      self.mStateList[i], i)
        self.scan(i)
    if self.mGrammar.finishObject() in self.mStateList[-1]:
        print('Parsing successful!')
    else:
        print('Parsing failed!')

EarleyParser.parse = earley_parse 
del earley_parse

### The Method `completeAndPredict`

The method `completeAndPredict(self, Qi, i)` computes a single step of the fixed-point iteration that computes the
set $Q_i$.  The argument `Qi` is the current approximation of $Q_i$.  The method returns the set `Qi` together with
all *Earley objects* that are found when the completion operation and the prediction operation are applied once to
`Qi`.  The operator `|` creates a new set, so `Qi` itself is not changed.  This is important, since the function
`fixpoint` compares the new set with `Qi` in order to decide whether the iteration has to go on.

In [ ]:
def completeAndPredict(self, Qi, i):
    return Qi | self.complete(Qi, i) | self.predict(Qi, i)

EarleyParser.completeAndPredict = completeAndPredict
del completeAndPredict

### The Method `complete`

The method `complete(self, Qi, i)` applies the completion operation to the set $Q_i$:
If we have
 - $\langle c \rightarrow \gamma \bullet, j\rangle \in Q_i$  and
 - $\langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_j$,
then the parser tried to parse the variable $c$ after having read `mString[:j+1]`
and we know that 
$$ c \Rightarrow^* \texttt{mString[j+1:i+1]}, $$
i.e. the parser has recognized $c$ after having read `mString[j+1:i+1]`.
Therefore the parser should proceed to recognize $\delta$ in the set $Q_i$.
Hence we add the *Earley object* $\langle a \rightarrow \beta c \bullet \delta,k\rangle$ to the set $Q_i$:
$$\langle c \rightarrow \gamma \bullet, j\rangle \in Q_i \wedge
  \langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_j \;\Rightarrow\;
          Q_i := Q_i \cup \bigl\{ \langle a \rightarrow \beta c \bullet \delta, k\rangle \bigr\}
$$
The method applies this operation once to the set `Qi`, which is the current approximation of $Q_i$, and returns
the set of all *Earley objects* that it finds.  It does not change `Qi`.

If $j = i$, then the *Earley object* $\langle a \rightarrow \beta \bullet c \delta, k\rangle$ is an element of $Q_i$
itself.  This can only happen if $c \Rightarrow^* \lambda$, i.e. if the grammar has $\lambda$-rules.  As
`self.mStateList[i]` is only updated after the fixed-point iteration has finished, the method has to look for this
*Earley object* in `Qi` in this case.

In [ ]:
def complete(self, Qi, i):
    Result = set()
    for obj in Qi:
        if obj.isComplete():
            c  = obj.mVariable
            j  = obj.mIndex
            Qj = Qi if j == i else self.mStateList[j]
            for waiting in Qj:
                if waiting.sameVar(c):
                    Result.add(waiting.moveDot())
    return Result
    
EarleyParser.complete = complete
del complete

### The Method `predict`

The method `self.predict(Qi, i)` applies the prediction operation to the set $Q_i$: 
If $\langle a \rightarrow \beta \bullet c \delta, k \rangle \in Q_i$, then
the parser tries to recognize $c\delta$ after having read `mString[:i+1]`.  To this end
it has to parse $c$ in the string `mString[i+1:]`.
Therefore, if $c \rightarrow \gamma$ is a rule of our grammar,
we add the *Earley object* $\langle c \rightarrow \bullet \gamma, i\rangle$ to the set $Q_i$:
$$ \langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_i 
       \wedge (c \rightarrow \gamma) \in R 
       \;\Rightarrow\;
       Q_i := Q_i \cup\bigl\{ \langle c \rightarrow \bullet\gamma, i\rangle\bigr\}.
$$
Like the method `complete`, the method `predict` applies this operation once to the set `Qi`, which is the
current approximation of $Q_i$, and returns the set of all *Earley objects* that it finds.  As the right hand side
$\gamma$ might start with a variable, a new *Earley object* can lead to further predictions.  These are found in
the next step of the fixed-point iteration.

In [ ]:
def predict(self, Qi, i):
    Result = set()
    for obj in Qi:
        c = obj.nextVar()
        if c is not None:
            for rule in self.mGrammar.mRules:
                if c == rule[0]:
                    Result.add(EarleyObject(c, (), tuple(rule[1:]), i))
    return Result

EarleyParser.predict = predict
del predict

### The Method `scan`

The method `self.scan(i)` applies the scanning operation to the set $Q_i$.

If $\langle a \rightarrow \beta \bullet T \gamma, k\rangle \in Q_i$ and $T$ is a token,
then the parser tries to recognize the right hand side of the grammar rule
$$ a \rightarrow \beta T \gamma$$ 
and after having read `mString[k+1:i+1]` it has already recognized  $\beta$.
If we now have `mString[i+1] == T`, then the parser still has to recognize $\gamma$ in `mString[i+2:]`.
Therefore, the *Earley object* $\langle a \rightarrow \beta T \bullet \gamma, k\rangle$ is added to
the set $Q_{i+1}$:
$$\langle a \rightarrow \beta \bullet T \gamma, k\rangle \in Q_i \wedge x_{i+1} = T
       \;\Rightarrow\;
       Q_{i+1} := Q_{i+1} \cup \bigl\{ \langle a \rightarrow \beta T \bullet \gamma, k\rangle \bigr\}
$$

In [ ]:
def earley_scan(self, i):
    Qi = self.mStateList[i]
    n  = len(self.mString) - 1 # mString[0] = None
    if i + 1 <= n:
        T = self.mString[i+1]
        for obj in Qi:
            if obj.scan(T):
                self.mStateList[i+1].add(obj.moveDot())

EarleyParser.scan = earley_scan
del earley_scan

## Testing the Parser

### The Function `tokenize`

The function `tokenize` transforms the string `s` that is to be parsed into a list of tokens.  Parentheses and
operators are returned as literals, i.e. enclosed in double quotes, since the grammar writes them this way.
See below for an example.

In [ ]:
def tokenize(s):
    '''Transform the string s into a list of tokens.  The string s
       is supposed to represent an arithmetic expression.
    '''
    lexSpec = r'''([ \t]+)        |  # blanks and tabs
                  ([1-9][0-9]*|0) |  # number
                  ([()])          |  # parentheses 
                  ([-+*/])        |  # arithmetical operators
                  (.)                # unrecognized character
               '''
    tokenList = re.findall(lexSpec, s, re.VERBOSE)
    result    = []
    for ws, number, parenthesis, operator, error in tokenList:
        if ws:        # skip blanks and tabs
            continue
        elif number:
            result += [ 'NUMBER' ]
        elif parenthesis:
            result += [ f'"{parenthesis}"' ]
        elif operator:
            result += [ f'"{operator}"' ]
        else:
            result += [ f'ERROR({error})']
    return result

In [ ]:
tokenize('1 + 2 * 3')

### The Function `test`

The function `test` takes two arguments.
- `file` is the name of a file containing a grammar,
- `word` is a string that should be parsed.

First, the function `read_grammar` reads the grammar from `file` and converts it into a list of rules, which is
passed to the constructor of the class `Grammar`.  Then `word` is tokenized.  Finally, `word` is parsed using
*Earley's algorithm*.

In [ ]:
def test(file, word):
    grammar   = Grammar(read_grammar(file))
    TokenList = tokenize(word)
    ep        = EarleyParser(grammar, TokenList)
    ep.parse()

In [ ]:
test('simple.g', '1 + 2 * 3')

## Further Tests

The parser handles parentheses, and it rejects strings that are not generated by the grammar.

In [ ]:
test('simple.g', '(1 + 2) * 3')

In [ ]:
test('simple.g', '1 + * 3')

Earley's algorithm works for every context-free grammar, even for an ambiguous grammar like the grammar in the
file `ambiguous.g`.

In [ ]:
Code(filename='ambiguous.g', language='larkita')

In [ ]:
test('ambiguous.g', '1 + 2 * 3')

The grammar in the file `lambda.g` describes the same expressions as the grammar in `simple.g`, but it avoids
left recursion.  Therefore, it needs $\lambda$-rules, i.e. rules with an empty right hand side.  For such a
grammar, a prediction can enable a completion in the same set $Q_i$:  If $c \rightarrow \lambda$ is a rule, then the
prediction operation adds the complete *Earley object* $\langle c \rightarrow \bullet, i\rangle$ to $Q_i$.  The
fixed-point iteration in the method `parse` takes care of this case, since it applies completion and prediction
until the set $Q_i$ does not change any more.

In [ ]:
Code(filename='lambda.g', language='larkita')

In [ ]:
test('lambda.g', '1 + 2 * 3')